# Basis-aware indices and coordinate-system notation

This notebook shows what the basis-awareness work (vibe 000067) made
available, one section per feature:

1. **Familiar coordinate-system notation** — an expansion prints in its frame's
   own letters (WCS $i, j, k$; cylindrical $\mathbf{e}_r, \mathbf{e}_\theta, \mathbf{e}_z$).
2. **Coordinates know their basis** — the same vector in two frames is
   *distinct*, yet a shared index still sums *across* frames.
3. **Basis-aware steps** — a step keyed to one frame acts only on that frame.
4. **Custom naming & display label** — give your own frame coordinate letters,
   vector symbols, and a frame marker (the primed index $\mathbf{e}_{i'}$).

`@` is the dot, `%` the cross, `*` the tensor product.

In [ ]:
import tender
import tender.basis as tb
import tender.derivation as td
from IPython.display import Math, display

def disp(*exprs):
    for e in exprs:
        display(Math(e.latex()))

ctx = tender.Context()
a = tender.tensor("a", rank=1, ctx=ctx)

## 1. Familiar coordinate-system notation

Expanding $a = a^i\,\mathbf{e}_i$ in a well-known frame and unrolling the sum to
concrete directions prints the result in that frame's coordinate letters — and,
for WCS, the classic standalone $i, j, k$ vectors.

In [ ]:
def expand_unroll(frame):
    e = tb.expand_in_basis(a, frame, tb.Variance.Covariant)
    return td.unroll_sums(td.canonicalize(e))

disp(
    expand_unroll(tb.wcs(ctx)),          # a_x i + a_y j + a_z k
    expand_unroll(tb.cylindrical(ctx)),  # a_r e_r + a_θ e_θ + a_z e_z
    expand_unroll(tb.spherical(ctx)),    # a_r e_r + a_θ e_θ + a_φ e_φ
)

## 2. Coordinates know their basis

The same invariant $a$ expanded in two different frames is **not** algebraically
equal — each coordinate carries its frame's identity.  A second frame is built
with a prime label so the two are visibly distinct in one term.

In [ ]:
wcs = tb.wcs(ctx)
p = tender.tensor("p", rank=1, ctx=ctx)
q = tender.tensor("q", rank=1, ctx=ctx)
s = tender.tensor("s", rank=1, ctx=ctx)
primed = tb.make_orthonormal_basis([p, q, s], tender.space_3d, label="'")

a_wcs    = td.canonicalize(tb.expand_in_basis(a, wcs,    tb.Variance.Covariant))
a_primed = td.canonicalize(tb.expand_in_basis(a, primed, tb.Variance.Covariant))
disp(a_wcs, a_primed)
print("algebraic_eq(WCS, WCS)   =", td.algebraic_eq(a_wcs, a_wcs))
print("algebraic_eq(WCS, primed)=", td.algebraic_eq(a_wcs, a_primed))

Yet a shared index still sums **across** frames — Einstein summation is
basis-blind.  That is what makes the rotation / two-point tensor
$R = \mathbf{e}_i \otimes \mathbf{e}'_i$ well formed (one summed index, two
frames).

In [ ]:
i = ctx.alloc_index()
rotation = td.canonicalize(wcs.covariant_vector(i) * primed.covariant_vector(i))
disp(rotation)

## 3. Basis-aware steps act only on their own frame

`simplify_basis_dot(frame)` contracts two basis vectors of *that* frame into a
Kronecker $\delta$.  The cross-frame overlap $\mathbf{e}_i^{\,\mathrm{WCS}}\cdot
\mathbf{e}_j^{\,\mathrm{primed}}$ is the change-of-basis matrix, not $\delta$, so
it is left untouched.

In [ ]:
j = ctx.alloc_index()
same    = td.canonicalize(tb.simplify_basis_dot(
              wcs.covariant_vector(i) @ wcs.covariant_vector(j), wcs))
overlap = tb.simplify_basis_dot(
              wcs.covariant_vector(i) @ primed.covariant_vector(j), wcs)
disp(same, overlap)

## 4. Custom naming & display label

The built-in systems are just frames with naming attached — you can configure
your own: `value_names` (coordinate letters), `vector_symbols` (standalone
direction symbols), and `label` (a frame marker on every index).

In [ ]:
u = tender.tensor("u", rank=1, ctx=ctx)
v = tender.tensor("v", rank=1, ctx=ctx)
w = tender.tensor("w", rank=1, ctx=ctx)
custom = tb.make_orthonormal_basis(
    [u, v, w], tender.space_3d,
    value_names=["\\xi", "\\eta", "\\zeta"],
    vector_symbols=["U", "V", "W"],
)
disp(td.unroll_sums(td.canonicalize(
    tb.expand_in_basis(a, custom, tb.Variance.Covariant))))

# The primed frame's label marks every index, so two frames are distinguishable
# even though both use the generic symbol e.
k = ctx.alloc_index()
disp(td.canonicalize(wcs.covariant_vector(k) * primed.covariant_vector(k)))